# Germline Variant Discovery and HLA Analysis

**Whole-Exome Sequencing (WES) Analysis**

This project presents a computational workflow for germline variant discovery and HLA-region analysis from whole-exome sequencing (WES) data.

The analysis includes:

1. Raw sequencing data acquisition and quality control
2. Read trimming and alignment to the human reference genome (hg38)
3. Post-alignment processing and quality assessment
4. Germline SNP and indel calling and filtering
5. HLA-region variant extraction and annotation
6. Exploration and interpretation of selected variants

**Tools:** SRA Toolkit, FastQC, Trimmomatic, BWA-MEM, SAMtools, Picard, GATK, BCFtools, and ANNOVAR.

In [1]:
%%bash
mkdir -p ~/germline_variant_analysis/data/raw
mkdir -p ~/germline_variant_analysis/reference
mkdir -p ~/germline_variant_analysis/results/fastqc/raw
mkdir -p ~/germline_variant_analysis/results/fastqc/trimmed
mkdir -p ~/germline_variant_analysis/results/trimmed
mkdir -p ~/germline_variant_analysis/results/alignment
mkdir -p ~/germline_variant_analysis/results/variants/raw
mkdir -p ~/germline_variant_analysis/results/variants/filtered
mkdir -p ~/germline_variant_analysis/results/variants/hla



# 1. Data Acquisition and Quality Control


## 1.1 Data Acquisition and FASTQ Conversion

The WES dataset used in this analysis is **SRR12083525**, obtained from the NCBI Sequence Read Archive (SRA).

The sequencing data are paired-end, meaning that each DNA fragment was sequenced from both ends, producing forward (R1) and reverse (R2) reads.

The downloaded SRA file was converted to paired-end FASTQ format using the SRA Toolkit. The `fasterq-dump` command with the `--split-files` option separates the paired reads into two FASTQ files:

- **R1:** forward reads (`SRR12083525_1.fastq`)
- **R2:** reverse reads (`SRR12083525_2.fastq`)

In [ ]:
# download the data set 
!prefetch SRR12083525

In [ ]:
!fasterq-dump \
    ~/germline_variant_analysis/data/raw/SRR12083525.sra \
    --split-files \
    -O ~/germline_variant_analysis/data/raw/

## 1.2 Raw Read Quality Control

The quality of the raw sequencing reads was assessed using FastQC before read trimming. FastQC was used to evaluate sequencing quality and identify potential issues such as low-quality bases, adapter contamination, and unusual sequence composition.

In [ ]:
!fastqc \
    ~/germline_variant_analysis/data/raw/SRR12083525_1.fastq \
    ~/germline_variant_analysis/data/raw/SRR12083525_2.fastq \
    -o ~/germline_variant_analysis/results/fastqc/raw

null
null
Started analysis of SRR12083525_1.fastq
Approx 5% complete for SRR12083525_1.fastq
Approx 10% complete for SRR12083525_1.fastq
Approx 15% complete for SRR12083525_1.fastq
Approx 20% complete for SRR12083525_1.fastq
Approx 25% complete for SRR12083525_1.fastq
Approx 30% complete for SRR12083525_1.fastq
Approx 35% complete for SRR12083525_1.fastq
Approx 40% complete for SRR12083525_1.fastq
Approx 45% complete for SRR12083525_1.fastq
Approx 50% complete for SRR12083525_1.fastq
Approx 55% complete for SRR12083525_1.fastq
Approx 60% complete for SRR12083525_1.fastq
Approx 65% complete for SRR12083525_1.fastq
Approx 70% complete for SRR12083525_1.fastq
Approx 75% complete for SRR12083525_1.fastq
Approx 80% complete for SRR12083525_1.fastq
Approx 85% complete for SRR12083525_1.fastq
Approx 90% complete for SRR12083525_1.fastq
Approx 95% complete for SRR12083525_1.fastq
Analysis complete for SRR12083525_1.fastq
Started analysis of SRR12083525_2.fastq
Approx 5% complete for SRR1208352

### FastQC Output

FastQC generates an HTML report and a ZIP archive for each input FASTQ file.
The HTML reports provide graphical summaries of sequencing quality, including
base quality, GC content, sequence duplication, and adapter content.

The ZIP archives were extracted to access the individual result files and
figures used for downstream visualization and interpretation.

In [ ]:
# Navigate to the FastQC results directory

%cd ~/germline_variant_analysis/results/fastqc/raw/

# Extract FastQC reports

!unzip -o SRR12083525_1_fastqc.zip
!unzip -o SRR12083525_2_fastqc.zip

~/germline_variant_analysis/results/fastqc
Archive:  SRR12083525_1_fastqc.zip
   creating: SRR12083525_1_fastqc/
   creating: SRR12083525_1_fastqc/Icons/
   creating: SRR12083525_1_fastqc/Images/
  inflating: SRR12083525_1_fastqc/Icons/fastqc_icon.png  
  inflating: SRR12083525_1_fastqc/Icons/warning.png  
  inflating: SRR12083525_1_fastqc/Icons/error.png  
  inflating: SRR12083525_1_fastqc/Icons/tick.png  
  inflating: SRR12083525_1_fastqc/summary.txt  
  inflating: SRR12083525_1_fastqc/Images/per_base_quality.svg  
  inflating: SRR12083525_1_fastqc/Images/per_base_quality.png  
  inflating: SRR12083525_1_fastqc/Images/per_sequence_quality.svg  
  inflating: SRR12083525_1_fastqc/Images/per_sequence_quality.png  
  inflating: SRR12083525_1_fastqc/Images/per_base_sequence_content.svg  
  inflating: SRR12083525_1_fastqc/Images/per_base_sequence_content.png  
  inflating: SRR12083525_1_fastqc/Images/per_sequence_gc_content.svg  
  inflating: SRR12083525_1_fastqc/Images/per_sequence_gc_con

### Visual Summary
### Per Base Sequence Quality
<table>
<tr>
<td><img src="results/fastqc/raw/SRR12083525_1_fastqc/Images/per_base_quality.png" width="400"></td>
<td><img src="results/fastqc/raw/SRR12083525_2_fastqc/Images/per_base_quality.png" width="400"></td>
</tr>
<tr><td align="center"><b>Forward Read</b></td><td align="center"><b>Reverse Read</b></td></tr>
</table>



### Per Base N Content
<table>
<tr>
<td><img src="results/fastqc/raw/SRR12083525_1_fastqc/Images/per_base_n_content.png" width="400"></td>
<td><img src="results/fastqc/raw/SRR12083525_2_fastqc/Images/per_base_n_content.png" width="400"></td>
</tr>
<tr><td align="center"><b>Forward Read</b></td><td align="center"><b>Reverse Read</b></td></tr>
</table>



### Adapter Content
<table>
<tr>
<td><img src="results/fastqc/raw/SRR12083525_1_fastqc/Images/adapter_content.png" width="400"></td>
<td><img src="results/fastqc/raw/SRR12083525_2_fastqc/Images/adapter_content.png" width="400"></td>
</tr>
<tr><td align="center"><b>Forward Read</b></td><td align="center"><b>Reverse Read</b></td></tr>
</table>

### Interpretation

FastQC indicated generally good overall sequencing quality for both paired-end reads. Most quality-control modules passed, including per-sequence quality scores, per-base sequence content, GC content, sequence length distribution, sequence duplication levels, overrepresented sequences, and adapter content.

The forward reads (R1) showed a warning for per-base sequence quality, while the reverse reads (R2) passed this module. Both R1 and R2 showed warnings for per-base N content, indicating the presence of positions with elevated proportions of undetermined bases.

Although adapter content did not trigger a warning, read trimming was performed to improve read quality and remove low-quality bases before downstream alignment and variant calling.

## 1.3 Read Trimming

The paired-end reads were trimmed using Trimmomatic to remove adapter sequences and low-quality bases. 

In [3]:
!trimmomatic PE -threads 4 \
    ~/germline_variant_analysis/data/raw/SRR12083525_1.fastq \
    ~/germline_variant_analysis/data/raw/SRR12083525_2.fastq \
    ~/germline_variant_analysis/results/trimmed/SRR12083525_1_paired.fq \
    ~/germline_variant_analysis/results/trimmed/SRR12083525_1_unpaired.fq \
    ~/germline_variant_analysis/results/trimmed/SRR12083525_2_paired.fq \
    ~/germline_variant_analysis/results/trimmed/SRR12083525_2_unpaired.fq \
    ILLUMINACLIP:/usr/share/trimmomatic/adapters/TruSeq3-PE.fa:2:30:10 \
    HEADCROP:3 \
    CROP:118 \
    LEADING:10 \
    TRAILING:10 \
    SLIDINGWINDOW:4:15 \
    MINLEN:36

TrimmomaticPE: Started with arguments:
 -threads 4 ~/germline_variant_analysis/data/raw/SRR12083525_1.fastq ~/germline_variant_analysis/data/raw/SRR12083525_2.fastq ~/germline_variant_analysis/results/trimmed/SRR12083525_1_paired.fq ~/germline_variant_analysis/results/trimmed/SRR12083525_1_unpaired.fq ~/germline_variant_analysis/results/trimmed/SRR12083525_2_paired.fq ~/germline_variant_analysis/results/trimmed/SRR12083525_2_unpaired.fq ILLUMINACLIP:/usr/share/trimmomatic/adapters/TruSeq3-PE.fa:2:30:10 HEADCROP:3 CROP:118 LEADING:10 TRAILING:10 SLIDINGWINDOW:4:15 MINLEN:36
Using PrefixPair: 'TACACTCTTTCCCTACACGACGCTCTTCCGATCT' and 'GTGACTGGAGTTCAGACGTGTGCTCTTCCGATCT'
ILLUMINACLIP: Using 1 prefix pairs, 0 forward/reverse sequences, 0 forward only sequences, 0 reverse only sequences
Quality encoding detected as phred33
Input Read Pairs: 29371176 Both Surviving: 25725293 (87.59%) Forward Only Surviving: 2484506 (8.46%) Reverse Only Surviving: 957605 (3.26%) Dropped: 203772 (0.69%)
Trimmom

### Trimmomatic Parameters

<table>
<tr>
<th>Parameter</th>
<th>Setting</th>
<th>Purpose</th>
</tr>

<tr>
<td><code>ILLUMINACLIP</code></td>
<td><code>2:30:10</code></td>
<td>Remove adapter sequences using the TruSeq3 paired-end adapter file</td>
</tr>

<tr>
<td><code>HEADCROP</code></td>
<td><code>3</code></td>
<td>Remove the first 3 bases of each read</td>
</tr>

<tr>
<td><code>CROP</code></td>
<td><code>118</code></td>
<td>Keep the first 118 bases after cropping</td>
</tr>

<tr>
<td><code>LEADING</code></td>
<td><code>10</code></td>
<td>Remove low-quality bases from the beginning of reads</td>
</tr>

<tr>
<td><code>TRAILING</code></td>
<td><code>10</code></td>
<td>Remove low-quality bases from the end of reads</td>
</tr>

<tr>
<td><code>SLIDINGWINDOW</code></td>
<td><code>4:15</code></td>
<td>Trim when the average quality in a 4-base window falls below 15</td>
</tr>

<tr>
<td><code>MINLEN</code></td>
<td><code>36</code></td>
<td>Discard reads shorter than 36 bases after trimming</td>
</tr>

</table>

### Trimming Results Summary

| Metric | Count | Percentage |
|---------|--------|-------------|
| Input Read Pairs | 29,371,176 | 100% |
| Both Surviving | 25,725,293 | 87.59% |
| Forward Only | 2,484,506 | 8.46% |
| Reverse Only | 957,605 | 3.26% |
| Dropped | 203,772 | 0.69% |

The majority of read pairs (87.59%) survived trimming as paired reads, while only 0.69% of read pairs were discarded entirely.

## 1.4 Post-Trimming Quality Control

FastQC was performed again on the trimmed paired-end reads to assess read quality after adapter removal and quality filtering.

In [ ]:
!fastqc \
    ~/germline_variant_analysis/results/trimmed/SRR12083525_1_paired.fq \
    ~/germline_variant_analysis/results/trimmed/SRR12083525_2_paired.fq \
    -o ~/germline_variant_analysis/results/fastqc/trimmed

null
null
Started analysis of SRR12083525_1_paired.fq
Approx 5% complete for SRR12083525_1_paired.fq
Approx 10% complete for SRR12083525_1_paired.fq
Approx 15% complete for SRR12083525_1_paired.fq
Approx 20% complete for SRR12083525_1_paired.fq
Approx 25% complete for SRR12083525_1_paired.fq
Approx 30% complete for SRR12083525_1_paired.fq
Approx 35% complete for SRR12083525_1_paired.fq
Approx 40% complete for SRR12083525_1_paired.fq
Approx 45% complete for SRR12083525_1_paired.fq
Approx 50% complete for SRR12083525_1_paired.fq
Approx 55% complete for SRR12083525_1_paired.fq
Approx 60% complete for SRR12083525_1_paired.fq
Approx 65% complete for SRR12083525_1_paired.fq
Approx 70% complete for SRR12083525_1_paired.fq
Approx 75% complete for SRR12083525_1_paired.fq
Approx 80% complete for SRR12083525_1_paired.fq
Approx 85% complete for SRR12083525_1_paired.fq
Approx 90% complete for SRR12083525_1_paired.fq
Approx 95% complete for SRR12083525_1_paired.fq
Analysis complete for SRR12083525_1

### Post-Trimming FastQC Results

FastQC was repeated on the paired reads after trimming. Overall sequencing quality improved, and the quality-control modules passed for both reads.

The only remaining warning was for sequence length distribution. This is expected because the trimming procedure removes bases from reads based on the specified trimming parameters and quality thresholds, resulting in reads with different lengths.

Overall, the post-trimming FastQC results indicate that the reads were suitable for downstream alignment.

In [ ]:
# Navigate to the FastQC results directory

%cd ~/germline_variant_analysis/results/fastqc/trimmed/

# Extract FastQC reports

!unzip -o SRR12083525_1_paired_fastqc.zip
!unzip -o SRR12083525_2_paired_fastqc.zip

~/germline_variant_analysis/results/fastqc/trimmed
Archive:  SRR12083525_1_paired_fastqc.zip
   creating: SRR12083525_1_paired_fastqc/
   creating: SRR12083525_1_paired_fastqc/Icons/
   creating: SRR12083525_1_paired_fastqc/Images/
  inflating: SRR12083525_1_paired_fastqc/Icons/fastqc_icon.png  
  inflating: SRR12083525_1_paired_fastqc/Icons/warning.png  
  inflating: SRR12083525_1_paired_fastqc/Icons/error.png  
  inflating: SRR12083525_1_paired_fastqc/Icons/tick.png  
  inflating: SRR12083525_1_paired_fastqc/summary.txt  
  inflating: SRR12083525_1_paired_fastqc/Images/per_base_quality.svg  
  inflating: SRR12083525_1_paired_fastqc/Images/per_base_quality.png  
  inflating: SRR12083525_1_paired_fastqc/Images/per_sequence_quality.svg  
  inflating: SRR12083525_1_paired_fastqc/Images/per_sequence_quality.png  
  inflating: SRR12083525_1_paired_fastqc/Images/per_base_sequence_content.svg  
  inflating: SRR12083525_1_paired_fastqc/Images/per_base_sequence_content.png  
  inflating: SRR12

Archive:  SRR12083525_2_paired_fastqc.zip
   creating: SRR12083525_2_paired_fastqc/
   creating: SRR12083525_2_paired_fastqc/Icons/
   creating: SRR12083525_2_paired_fastqc/Images/
  inflating: SRR12083525_2_paired_fastqc/Icons/fastqc_icon.png  
  inflating: SRR12083525_2_paired_fastqc/Icons/warning.png  
  inflating: SRR12083525_2_paired_fastqc/Icons/error.png  
  inflating: SRR12083525_2_paired_fastqc/Icons/tick.png  
  inflating: SRR12083525_2_paired_fastqc/summary.txt  
  inflating: SRR12083525_2_paired_fastqc/Images/per_base_quality.svg  
  inflating: SRR12083525_2_paired_fastqc/Images/per_base_quality.png  
  inflating: SRR12083525_2_paired_fastqc/Images/per_sequence_quality.svg  
  inflating: SRR12083525_2_paired_fastqc/Images/per_sequence_quality.png  
  inflating: SRR12083525_2_paired_fastqc/Images/per_base_sequence_content.svg  
  inflating: SRR12083525_2_paired_fastqc/Images/per_base_sequence_content.png  
  inflating: SRR12083525_2_paired_fastqc/Images/per_sequence_gc_conte

### Visual Summary

### Per Base Sequence Quality




<table>
<tr>
<th></th>
<th>Before Trimming</th>
<th>After Trimming</th>
</tr>

<tr>
<td><b>Forward Read (R1)</b></td>
<td><img src="results/fastqc/raw/SRR12083525_1_fastqc/Images/per_base_quality.png" width="400"></td>
<td><img src="results/fastqc/trimmed/SRR12083525_1_paired_fastqc/Images/per_base_quality.png" width="400"></td>
</tr>

<tr>
<td><b>Reverse Read (R2)</b></td>
<td><img src="results/fastqc/raw/SRR12083525_2_fastqc/Images/per_base_quality.png" width="400"></td>
<td><img src="results/fastqc/trimmed/SRR12083525_2_paired_fastqc/Images/per_base_quality.png" width="400"></td>
</tr>
</table>

### Per Base N Content

<table>
<tr>
<th></th>
<th>Before Trimming</th>
<th>After Trimming</th>
</tr>

<tr>
<td><b>Forward Read (R1)</b></td>
<td><img src="results/fastqc/raw/SRR12083525_1_fastqc/Images/per_base_n_content.png" width="400"></td>
<td><img src="results/fastqc/trimmed/SRR12083525_1_paired_fastqc/Images/per_base_n_content.png" width="400"></td>
</tr>

<tr>
<td><b>Reverse Read (R2)</b></td>
<td><img src="results/fastqc/raw/SRR12083525_2_fastqc/Images/per_base_n_content.png" width="400"></td>
<td><img src="results/fastqc/trimmed/SRR12083525_2_paired_fastqc/Images/per_base_n_content.png" width="400"></td>
</tr>
</table>

### Post-Trimming FastQC Results

FastQC was repeated on the paired reads after trimming. Overall sequencing quality improved, and the quality-control modules passed for both reads.

The only remaining warning was for sequence length distribution. This is expected because the trimming procedure removes bases from reads based on the specified trimming parameters and quality thresholds, resulting in reads with different lengths.

Overall, the post-trimming FastQC results indicate that the reads were suitable for downstream alignment.

# 2. Read Alignment and Post-Processing

In this section, the cleaned and trimmed FASTQ files are aligned to the human reference genome (hg38) using BWA-MEM.

The alignment workflow includes:

1. Preparing the reference genome (FASTA, index, and sequence dictionary)
2. Indexing the reference with BWA
3. Aligning paired-end reads to hg38
4. Converting SAM to BAM, sorting, and indexing
5. Generating alignment statistics using `samtools flagstat`

## 2.1 Prepare the Reference Genome


The reference FASTA can be obtained from the UCSC Genome Browser:

[UCSC hg38 reference genome downloads](https://hgdownload.gi.ucsc.edu/goldenPath/hg38/bigZips/)

The FASTA used in this analysis is stored as `reference/hg38.fa`.


The reference genome was prepared for downstream alignment and variant calling by generating:

- a BWA index for read alignment
- a sequence dictionary required by GATK

The BWA index consists of the following files:

- `hg38.fa.amb`
- `hg38.fa.ann`
- `hg38.fa.bwt`
- `hg38.fa.pac`
- `hg38.fa.sa`

The sequence dictionary is stored as `hg38.dict`.

In [1]:
# Create BWA index
!bwa index ~/germline_variant_analysis/reference/hg38.fa

[bwa_index] Pack FASTA... 23.88 sec
[bwa_index] Construct BWT for the packed sequence...
[BWTIncCreate] textLength=6418572210, availableWord=463634060
[BWTIncConstructFromPacked] 10 iterations done. 99999986 characters processed.
[BWTIncConstructFromPacked] 20 iterations done. 199999986 characters processed.
[BWTIncConstructFromPacked] 30 iterations done. 299999986 characters processed.
[BWTIncConstructFromPacked] 40 iterations done. 399999986 characters processed.
[BWTIncConstructFromPacked] 50 iterations done. 499999986 characters processed.
[BWTIncConstructFromPacked] 60 iterations done. 599999986 characters processed.
[BWTIncConstructFromPacked] 70 iterations done. 699999986 characters processed.
[BWTIncConstructFromPacked] 80 iterations done. 799999986 characters processed.
[BWTIncConstructFromPacked] 90 iterations done. 899999986 characters processed.
[BWTIncConstructFromPacked] 100 iterations done. 999999986 characters processed.
[BWTIncConstructFromPacked] 110 iterations done. 

In [ ]:
# create dictionary
!gatk CreateSequenceDictionary \
    -R ~/germline_variant_analysis/reference/hg38.fa

Using GATK jar ~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar ~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar CreateSequenceDictionary -R ~/germline_variant_analysis/reference/hg38.fa
INFO	2026-09-12 20:06:22	CreateSequenceDictionary	Output dictionary will be written in ~/germline_variant_analysis/reference/hg38.dict
20:06:22.777 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
[Sat Sep 12 20:06:22 IRST 2026] CreateSequenceDictionary --REFERENCE ~/germline_variant_analysis/reference/hg38.fa --TRUNCATE_NAMES_AT_WHITESPACE true --NUM_SEQUENCES 2147483647 --VERBOSITY INFO --QUIET false --VALIDATION_STRINGENCY STRICT --COMPRESSION_LEVEL 2 --MAX_RECORDS_IN_RAM 500000 --CREATE_INDEX false --CREATE_MD5_FILE fa

In [4]:
!samtools faidx ~/germline_variant_analysis/reference/hg38.fa

## 2.2 Align Paired-End Reads Using BWA-MEM

The trimmed paired-end reads were aligned to the hg38 reference genome using BWA-MEM.

Read-group information (`@RG`) was included in the alignment command because it is required for downstream processing with Picard and GATK.

**Read Group Fields:**

- `ID=1` → read group identifier
- `SM=SRR12083525` → sample name
- `PL=ILLUMINA` → sequencing platform

In [3]:
!bwa mem -t 4 \
    -R "@RG\tID:1\tSM:SRR12083525\tPL:ILLUMINA" \
    ~/germline_variant_analysis/reference/hg38.fa \
    ~/germline_variant_analysis/results/trimmed/SRR12083525_1_paired.fq \
    ~/germline_variant_analysis/results/trimmed/SRR12083525_2_paired.fq \
    > ~/germline_variant_analysis/results/alignment/SRR12083525_rg.sam

[M::bwa_idx_load_from_disk] read 0 ALT contigs
[M::process] read 343548 sequences (40000188 bp)...
[M::process] read 342442 sequences (40000090 bp)...
[M::mem_pestat] # candidate unique pairs for (FF, FR, RF, RR): (8, 147693, 0, 4)
[M::mem_pestat] skip orientation FF as there are not enough pairs
[M::mem_pestat] analyzing insert size distribution for orientation FR...
[M::mem_pestat] (25, 50, 75) percentile: (176, 236, 314)
[M::mem_pestat] low and high boundaries for computing mean and std.dev: (1, 590)
[M::mem_pestat] mean and std.dev: (252.90, 98.18)
[M::mem_pestat] low and high boundaries for proper pairs: (1, 728)
[M::mem_pestat] skip orientation RF as there are not enough pairs
[M::mem_pestat] skip orientation RR as there are not enough pairs
[M::mem_process_seqs] Processed 343548 reads in 484.754 CPU sec, 234.221 real sec
[M::process] read 342860 sequences (40000043 bp)...
[M::mem_pestat] # candidate unique pairs for (FF, FR, RF, RR): (14, 118629, 1, 2)
[M::mem_pestat] analyzing 

## 2.3 Convert, Sort, and Index Alignments

The SAM alignment file was converted to BAM format using SAMtools. BAM is a compressed binary format that is more efficient for downstream processing.

The BAM file was then sorted by genomic coordinates and indexed to allow efficient access to specific genomic regions.

In [4]:
!samtools view -b \
    ~/germline_variant_analysis/results/alignment/SRR12083525_rg.sam \
    | samtools sort \
    -o ~/germline_variant_analysis/results/alignment/SRR12083525_rg_sorted.bam

!samtools index \
    ~/germline_variant_analysis/results/alignment/SRR12083525_rg_sorted.bam

[bam_sort_core] merging from 20 files and 1 in-memory blocks...


## 2.4 Alignment Evaluation

The alignment was evaluated using `samtools flagstat`, which summarizes key alignment statistics, including the number of reads mapped to the reference genome and the proportion of properly paired reads.

In [5]:
!samtools flagstat \
    ~/germline_variant_analysis/results/alignment/SRR12083525_rg_sorted.bam

51090460 + 0 in total (QC-passed reads + QC-failed reads)
51066390 + 0 primary
0 + 0 secondary
24070 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
51088164 + 0 mapped (100.00% : N/A)
51064094 + 0 primary mapped (100.00% : N/A)
51066390 + 0 paired in sequencing
25533195 + 0 read1
25533195 + 0 read2
50617000 + 0 properly paired (99.12% : N/A)
51061812 + 0 with itself and mate mapped
2282 + 0 singletons (0.00% : N/A)
401098 + 0 with mate mapped to a different chr
387594 + 0 with mate mapped to a different chr (mapQ>=5)


### Alignment Summary

The alignment produced 51,090,460 reads in total. All reads were mapped to the reference genome, and 99.12% of reads were properly paired.

| Metric | Result |
|---|---:|
| Total reads | 51,090,460 |
| Mapped reads | 51,088,164 (100.00%) |
| Properly paired | 50,617,000 (99.12%) |
| Singletons | 2,282 (0.00%) |
| Supplementary alignments | 24,070 |
| Duplicates | 0 |

The very high mapping rate and proportion of properly paired reads indicate successful alignment of the trimmed reads to the hg38 reference genome. The low number of singletons also indicates that most read pairs remained consistently aligned.

### 2.5 Mark PCR Duplicates

PCR amplification introduces duplicate reads that originate from the same DNA fragment.  
These do not add independent biological information and should be marked before variant calling.

Picard `MarkDuplicates` identifies duplicate fragments based on their alignment coordinates and marks them in the BAM file.

The command below produces:

- a duplicate-marked BAM file (`SRR12083525_dedup.bam`)
- an index (`.bai`)
- a duplication metrics summary (`SRR12083525_dedup_metrics.txt`)

In [1]:
!java -jar ~/tools/picard.jar MarkDuplicates \
    I=~/germline_variant_analysis/results/alignment/SRR12083525_rg_sorted.bam \
    O=~/germline_variant_analysis/results/alignment/SRR12083525_dedup.bam \
    M=~/germline_variant_analysis/results/alignment/SRR12083525_dedup_metrics.txt \
    CREATE_INDEX=true \
    VALIDATION_STRINGENCY=SILENT

INFO	2026-09-13 12:11:11	MarkDuplicates	

********** NOTE: Picard's command line syntax is changing.
**********
********** For more information, please see:
********** 
https://github.com/broadinstitute/picard/wiki/Command-Line-Syntax-Transition-For-Users-(Pre-Transition)
**********
********** The command line looks like this in the new syntax:
**********
**********    MarkDuplicates -I ~/germline_variant_analysis/results/alignment/SRR12083525_rg_sorted.bam -O ~/germline_variant_analysis/results/alignment/SRR12083525_dedup.bam -M ~/germline_variant_analysis/results/alignment/SRR12083525_dedup_metrics.txt -CREATE_INDEX true -VALIDATION_STRINGENCY SILENT
**********


12:11:11.820 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:~/tools/picard.jar!/com/intel/gkl/native/libgkl_compression.so
[Sun Sep 13 12:11:11 IRST 2026] MarkDuplicates INPUT=[~/germline_variant_analysis/results/alignment/SRR12083525_rg_sorted.bam] OUTPUT=~/germline_variant_analysis/results/alignmen

In [2]:
!cat ~/germline_variant_analysis/results/alignment/SRR12083525_dedup_metrics.txt

## htsjdk.samtools.metrics.StringHeader
# MarkDuplicates INPUT=[~/germline_variant_analysis/results/alignment/SRR12083525_rg_sorted.bam] OUTPUT=~/germline_variant_analysis/results/alignment/SRR12083525_dedup.bam METRICS_FILE=~/germline_variant_analysis/results/alignment/SRR12083525_dedup_metrics.txt VALIDATION_STRINGENCY=SILENT CREATE_INDEX=true    MAX_SEQUENCES_FOR_DISK_READ_ENDS_MAP=50000 MAX_FILE_HANDLES_FOR_READ_ENDS_MAP=8000 SORTING_COLLECTION_SIZE_RATIO=0.25 TAG_DUPLICATE_SET_MEMBERS=false REMOVE_SEQUENCING_DUPLICATES=false TAGGING_POLICY=DontTag CLEAR_DT=true DUPLEX_UMI=false FLOW_MODE=false FLOW_DUP_STRATEGY=FLOW_QUALITY_SUM_STRATEGY FLOW_USE_END_IN_UNPAIRED_READS=false FLOW_USE_UNPAIRED_CLIPPED_END=false FLOW_UNPAIRED_END_UNCERTAINTY=0 FLOW_UNPAIRED_START_UNCERTAINTY=0 FLOW_SKIP_FIRST_N_FLOWS=0 FLOW_Q_IS_KNOWN_END=false FLOW_EFFECTIVE_QUALITY_THRESHOLD=15 ADD_PG_TAG_TO_READS=true REMOVE_DUPLICATES=false ASSUME_SORTED=false DUPLICATE_SCORING_STRATEGY=SUM_OF_BASE_QUALITIES PROGR

### Duplicate Marking Results

Picard `MarkDuplicates` identified a small fraction of reads as PCR duplicates.

| Metric | Result |
|---|---:|
| Read pairs examined | 25,530,906 |
| Read-pair duplicates | 176,536 |
| Optical duplicates | 0 |
| Percent duplication | 0.69% |
| Unmapped reads | 2,296 |

Only 0.69% of the reads were identified as duplicates, indicating a low level of PCR duplication in this dataset.

# 3. Germline Variant Calling and Filtering

In this section, germline single-nucleotide variants (SNVs) and small insertions/deletions (indels) are identified from the processed WES alignment.

The workflow includes:

1. Germline variant calling with GATK HaplotypeCaller
2. Variant quality assessment and filtering


### 3.1 Germline Variant Calling with GATK HaplotypeCaller

GATK HaplotypeCaller was used to identify germline variants from the duplicate-marked BAM file.

HaplotypeCaller performs local re-assembly of reads in regions containing potential variation and generates an unfiltered VCF file containing the called variants.

In [ ]:
!gatk HaplotypeCaller \
    -R ~/germline_variant_analysis/reference/hg38.fa \
    -I ~/germline_variant_analysis/results/alignment/SRR12083525_dedup.bam \
    -O ~/germline_variant_analysis/results/variants/raw/raw_variants.vcf.gz

Using GATK jar ~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar ~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar HaplotypeCaller -R ~/germline_variant_analysis/reference/hg38.fa -I ~/germline_variant_analysis/results/alignment/SRR12083525_dedup.bam -O ~/germline_variant_analysis/results/variants/raw_variants.vcf.gz
13:26:06.204 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
13:26:12.298 INFO  HaplotypeCaller - ------------------------------------------------------------
13:26:12.302 INFO  HaplotypeCaller - The Genome Analysis Toolkit (GATK) v4.6.2.0
13:26:12.302 INFO  HaplotypeCaller - For support and documentation go to https://software.broadinstitute.org/gatk/
13:26:12.303 INFO  HaplotypeCaller - Execu

### 3.2 Variant Filtering

Raw variant calls may include technical artifacts. SNPs and indels were therefore separated and filtered independently using GATK.



### Separating SNPs and INDELs

The raw VCF contains both single-nucleotide polymorphisms (SNPs) and small insertions/deletions (INDELs). These variant types are separated using GATK `SelectVariants` so that they can be filtered using appropriate quality criteria.

The first command extracts SNPs, while the second extracts INDELs. The resulting files are used for separate filtering in the next steps.

In [ ]:
!gatk SelectVariants \
    -R ~/germline_variant_analysis/reference/hg38.fa \
    -V ~/germline_variant_analysis/results/variants/raw/raw_variants.vcf.gz \
    --select-type-to-include SNP \
    -O ~/germline_variant_analysis/results/variants/raw/raw_snps.vcf.gz

!gatk SelectVariants \
    -R ~/germline_variant_analysis/reference/hg38.fa \
    -V ~/germline_variant_analysis/results/variants/raw/raw_variants.vcf.gz \
    --select-type-to-include INDEL \
    -O ~/germline_variant_analysis/results/variants/raw/raw_indels.vcf.gz

Using GATK jar ~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar ~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar SelectVariants -R ~/germline_variant_analysis/reference/hg38.fa -V ~/germline_variant_analysis/results/variants/raw_variants.vcf.gz --select-type-to-include SNP -O ~/germline_variant_analysis/results/variants/raw_snps.vcf.gz
15:08:37.727 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
15:08:37.924 INFO  SelectVariants - ------------------------------------------------------------
15:08:37.927 INFO  SelectVariants - The Genome Analysis Toolkit (GATK) v4.6.2.0
15:08:37.927 INFO  SelectVariants - For support and documentation go to https://software.broadinstitute.org/gatk/
15:08:37.927 INFO  Selec

###  SNP Filtering

The SNP variants were filtered using GATK `VariantFiltration` based on several quality metrics.

The following thresholds were applied:

- **QD < 2.0** → low variant confidence relative to sequencing depth
- **QUAL < 30** → low variant quality
- **MQ < 40** → low mapping quality
- **DP < 10 or DP > 200** → insufficient or unusually high sequencing depth

Variants that meet any of these criteria are flagged as low quality rather than immediately removed. Variants that pass all filters are retained for downstream analysis.

In [ ]:
!gatk VariantFiltration \
    -R ~/germline_variant_analysis/reference/hg38.fa \
    -V ~/germline_variant_analysis/results/variants/raw/raw_snps.vcf.gz \
    --filter-name "QD2" --filter-expression "QD < 2.0" \
    --filter-name "QUAL30" --filter-expression "QUAL < 30.0" \
    --filter-name "MQ40" --filter-expression "MQ < 40.0" \
    --filter-name "DPFilter" --filter-expression "DP < 10 || DP > 200" \
    -O ~/germline_variant_analysis/results/variants/filtered/filtered_snps.vcf.gz

Using GATK jar ~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar ~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar VariantFiltration -R ~/germline_variant_analysis/reference/hg38.fa -V ~/germline_variant_analysis/results/variants/raw_snps.vcf.gz --filter-name QD2 --filter-expression QD < 2.0 --filter-name QUAL30 --filter-expression QUAL < 30.0 --filter-name MQ40 --filter-expression MQ < 40.0 --filter-name DPFilter --filter-expression DP < 10 || DP > 200 -O ~/germline_variant_analysis/results/variants/filtered_snps.vcf.gz
15:12:57.240 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
15:12:57.408 INFO  VariantFiltration - ------------------------------------------------------------
15:12:57.413 INFO  VariantFi

### INDEL Filtering

The INDEL variants were filtered separately using GATK `VariantFiltration`.

The following thresholds were applied:

- **QD < 2.0** → low variant confidence relative to sequencing depth
- **QUAL < 30** → low variant quality
- **DP < 10 or DP > 200** → insufficient or unusually high sequencing depth


In [ ]:
!gatk VariantFiltration \
    -R ~/germline_variant_analysis/reference/hg38.fa \
    -V ~/germline_variant_analysis/results/variants/raw/raw_indels.vcf.gz \
    --filter-name "QD2" --filter-expression "QD < 2.0" \
    --filter-name "QUAL30" --filter-expression "QUAL < 30.0" \
    --filter-name "DPFilter" --filter-expression "DP < 10 || DP > 200" \
    -O ~/germline_variant_analysis/results/variants/filtered/filtered_indels.vcf.gz

Using GATK jar ~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar ~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar VariantFiltration -R ~/germline_variant_analysis/reference/hg38.fa -V ~/germline_variant_analysis/results/variants/raw_indels.vcf.gz --filter-name QD2 --filter-expression QD < 2.0 --filter-name QUAL30 --filter-expression QUAL < 30.0 --filter-name DPFilter --filter-expression DP < 10 || DP > 200 -O ~/germline_variant_analysis/results/variants/filtered_indels.vcf.gz
15:14:55.337 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
15:14:55.519 INFO  VariantFiltration - ------------------------------------------------------------
15:14:55.522 INFO  VariantFiltration - The Genome Analysis Toolkit (GATK)

###  Merging Filtered Variants

The filtered SNP and INDEL VCF files were combined into a single VCF file using `bcftools concat`.

The merged file contains both variant types and will be used for the final extraction of variants that passed all quality filters.

In [ ]:
%%bash

bcftools concat \
    ~/germline_variant_analysis/results/variantsfiltered/filtered_snps.vcf.gz \
    ~/germline_variant_analysis/results/variants/filtered/filtered_indels.vcf.gz \
    -a -O z \
    -o ~/germline_variant_analysis/results/variants/filtered/merged_filtered.vcf.gz

gatk IndexFeatureFile \
    -I ~/germline_variant_analysis/results/variants/filtered/merged_filtered.vcf.gz

Checking the headers and starting positions of 2 files
Using GATK jar ~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar ~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar IndexFeatureFile -I ~/germline_variant_analysis/results/variants/merged_filtered.vcf.gz
15:16:28.023 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
15:16:28.201 INFO  IndexFeatureFile - ------------------------------------------------------------
15:16:28.204 INFO  IndexFeatureFile - The Genome Analysis Toolkit (GATK) v4.6.2.0
15:16:28.205 INFO  IndexFeatureFile - For support and documentation go to https://software.broadinstitute.org/gatk/
15:16:28.205 INFO  IndexFeatureFile - Executing as user@host on Linux v5.10.16.3-microsoft-standard-

Tool returned:
~/germline_variant_analysis/results/variants/merged_filtered.vcf.gz.tbi


###  Extracting PASS Variants

The merged VCF contains both variants that passed the filters and variants that were flagged by one or more filtering criteria.

GATK `SelectVariants` was used with `--exclude-filtered` to remove the filtered variants and retain only variants with a `PASS` status.

The resulting VCF contains the high-confidence variants used for downstream analysis.

In [ ]:
!gatk SelectVariants \
    -V ~/germline_variant_analysis/results/variants/filtered/merged_filtered.vcf.gz \
    --exclude-filtered \
    -O ~/germline_variant_analysis/results/variants/filtered/merged_filtered_PASS.vcf.gz

Using GATK jar ~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar ~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar SelectVariants -V ~/germline_variant_analysis/results/variants/merged_filtered.vcf.gz --exclude-filtered -O ~/germline_variant_analysis/results/variants/merged_filtered_PASS.vcf.gz
15:17:36.166 INFO  NativeLibraryLoader - Loading libgkl_compression.so from jar:file:~/gatk-4.6.2.0/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.so
15:17:36.347 INFO  SelectVariants - ------------------------------------------------------------
15:17:36.351 INFO  SelectVariants - The Genome Analysis Toolkit (GATK) v4.6.2.0
15:17:36.351 INFO  SelectVariants - For support and documentation go to https://software.broadinstitute.org/gatk/
15:17:36.352 INFO  SelectVariants - Executing as user@host on Linux 

###  Variant Statistics 

Basic statistics were calculated for the final set of PASS variants, including the total number of variants, the number of SNPs and INDELs, and the transition-to-transversion (Ti/Tv) ratio.



In [ ]:
%%bash

# Total variants
bcftools view -H ~/germline_variant_analysis/results/variants/filtered/merged_filtered_PASS.vcf.gz | wc -l

# SNP count
bcftools view -H -v snps ~/germline_variant_analysis/results/variants/filtered/merged_filtered_PASS.vcf.gz | wc -l

# INDEL count
bcftools view -H -v indels ~/germline_variant_analysis/results/variants/filtered/merged_filtered_PASS.vcf.gz | wc -l

# Generate detailed variant statistics
bcftools stats \
    ~/germline_variant_analysis/results/variants/filtered/merged_filtered_PASS.vcf.gz \
    > ~/germline_variant_analysis/results/variants/filtered/stats.txt

112495
97268
15227


### Variant Statistics table



| Metric | Result |
|---|---:|
| Total PASS variants | 112,495 |
| SNPs | 97,268 |
| INDELs | 15,227 |
| Transitions | 68,021 |
| Transversions | 29,267 |
| Ti/Tv ratio | 2.32 |



# 4. HLA-Region Analysis

The human leukocyte antigen (HLA) region is a highly polymorphic region of the human genome involved in immune recognition and antigen presentation.

In this section, variants located within HLA genes are extracted from the final set of high-confidence germline variants.

The workflow includes:

1. Defining HLA genomic intervals
2. Extracting HLA-region variants from the filtered VCF
3. Indexing and inspecting the extracted variants
4. Annotating HLA variants using ANNOVAR

## 4.1 HLA variants Extraction
To identify variants located in HLA genes, genomic intervals corresponding to the HLA genes are required.

### Creating the HLA Gene BED File

HLA gene coordinates were extracted from the **GENCODE v45 annotation (GTF)**.

The annotation was searched for genes whose `gene_name` begins with `HLA-`. Only records with the feature type `gene` were retained. For each HLA gene, the chromosome, genomic start, genomic end, and gene name were written to a BED file.

Because BED coordinates are 0-based and GTF coordinates are 1-based, the GTF start coordinate was converted by subtracting 1.

The resulting file is stored as:

`reference/HLA_all_genes.bed`

In [1]:
%%bash
grep 'gene_name "HLA-' ~/germline_variant_analysis/reference/gencode.v45.annotation.gtf | \
awk '$3=="gene" {
    match($0, /gene_name "([^"]+)"/, arr);
    gene=arr[1];
    print $1 "\t" $4-1 "\t" $5 "\t" gene
}' > ~/germline_variant_analysis/reference/HLA_all_genes.bed

In [2]:
!head -n 10 ~/germline_variant_analysis/reference/HLA_all_genes.bed

chr6	29722774	29738528	HLA-F
chr6	29726600	29749049	HLA-F-AS1
chr6	29790953	29797811	HLA-V
chr6	29792233	29793136	HLA-V
chr6	29800414	29802425	HLA-P
chr6	29826966	29831125	HLA-G
chr6	29887751	29890482	HLA-H
chr6	29896653	29897786	HLA-T
chr6	29926458	29929232	HLA-K
chr6	29934100	29934286	HLA-U


### Generating VCF File of HLA variants

The HLA gene intervals defined in `HLA_all_genes.bed` were used to extract variants from the final set of high-confidence (`PASS`) variants.

`bcftools view -R` was used to select variants overlapping the genomic intervals specified in the BED file. The extracted variants were saved as a compressed VCF file for further analysis.

In [ ]:
!bcftools view \
    -R ~/germline_variant_analysis/reference/HLA_all_genes.bed \
    -Oz \
    -o ~/germline_variant_analysis/results/variants/hla/HLA_variants.vcf.gz \
    ~/germline_variant_analysis/results/variants/filtered/merged_filtered_PASS.vcf.gz

###  Indexing the HLA Variant VCF

The extracted HLA variants were stored as a compressed VCF file. The VCF was indexed using `bcftools` to enable efficient access to variants at specific genomic positions without reading the entire file.

The index is required by many downstream genomic tools when querying compressed VCF files by genomic region.

In [ ]:
!bcftools index \
    ~/germline_variant_analysis/results/variants/hla/HLA_variants.vcf.gz

In [ ]:
!bcftools view -H \
    ~/germline_variant_analysis/results/variants/hla/HLA_variants.vcf.gz \
    | wc -l

37


###  HLA Variant Statistics

Basic statistics were calculated for the HLA-region variants, including the total number of variants, the number of SNPs and INDELs, and detailed variant statistics generated with `bcftools stats`.

These statistics provide an overview of the types of variants identified within the HLA genomic intervals before downstream annotation.

In [ ]:
%%bash

# Total variants
bcftools view -H \
    ~/germline_variant_analysis/results/variants/hla/HLA_variants.vcf.gz \
    | wc -l

# SNP count
bcftools view -H -v snps \
    ~/germline_variant_analysis/results/variants/hla/HLA_variants.vcf.gz \
    | wc -l

# INDEL count
bcftools view -H -v indels \
    ~/germline_variant_analysis/results/variants/hla/HLA_variants.vcf.gz \
    | wc -l

# Generate detailed variant statistics
bcftools stats \
    ~/germline_variant_analysis/results/variants/hla/HLA_variants.vcf.gz \
    > ~/germline_variant_analysis/results/variants/hla/HLA_stats.txt

37
35
2


### HLA Variants Summary Table

| Metric                       | Value        |
|------------------------------|--------------|
| Total variants               | 37           |
| SNP count                    | 35           |
| INDEL count                  | 2            |
| Transition count (ts)        | 23           |
| Transversion count (tv)      | 12     `     |
| Ti/Tv ratio                  | 1.92         |


## 4.2 HLA Variant Annotation 

The HLA-region variants were annotated using **ANNOVAR** to add functional and database-based information to the variant calls.

The annotation workflow uses the following databases:

- **refGene** — gene-based functional annotation
- **ClinVar** — clinical significance annotations
- **gnomAD exome** — population allele frequency information



In [ ]:
!perl ~/annovar/table_annovar.pl \
    ~/germline_variant_analysis/results/variants/hla/HLA_variants.vcf.gz \
    ~/annovar/humandb/ \
    -buildver hg38 \
    -out ~/germline_variant_analysis/results/variants/hla/HLA_annotated \
    -remove \
    -protocol refGene,clinvar_20220320,gnomad211_exome,dbnsfp42a \
    -operation g,f,f,f \
    -nastring . \
    -vcfinput

NOTICE: the --polish argument is set ON automatically (use --nopolish to change this behavior)

NOTICE: Running with system command <convert2annovar.pl  -includeinfo -allsample -withfreq -format vcf4 ~/germline_variant_analysis/results/variants/HLA_variants.vcf.gz > ~/germline_variant_analysis/results/variants/HLA_annotated.avinput>
NOTICE: Finished reading 528 lines from VCF file
NOTICE: A total of 37 locus in VCF file passed QC threshold, representing 35 SNPs (23 transitions and 12 transversions) and 2 indels/substitutions
NOTICE: Finished writing allele frequencies based on 35 SNP genotypes (23 transitions and 12 transversions) and 2 indels/substitutions for 1 samples

NOTICE: Running with system command <~/annovar/table_annovar.pl ~/germline_variant_analysis/results/variants/HLA_annotated.avinput ~/annovar/humandb/ -buildver hg38 -outfile ~/germline_variant_analysis/results/variants/HLA_annotated -remove -protocol refGene,clinvar_20220320,gnomad211_exome,dbnsfp42a -operation g,f,f,

### HLA Annotation Summary


ANNOVAR functional annotations were examined to characterize the consequences of the HLA-region variants.

| Gene | Total variants | Exonic variants | Nonsynonymous | Frameshift |
|---|---:|---:|---:|---:|
| HLA-A | 5 | 4 | 3 | 0 |
| HLA-H | 1 | 0 | 0 | 0 |
| HLA-DRB1 | 5 | 5 | 1 | 2 |
| HLA-DQA1 | 22 | 5 | 3 | 0 |
| HLA-DQB1 | 3 | 3 | 2 | 0 |
| HLA-DQB1-AS1 | 1 | 0 | 0 | 0 |



## 4.3 HLA Variants Biological Interpretation

### HLA molecular structure

**MHC Class I**: It is composed of a heavy chain and an invariant beta-2-microglobulin (β2m) that presents peptides to T cells. HLA-I molecules possess six distinct binding pockets within the binding cleft termed as pockets A–F which allow peptide residue side chains to anchor or bind deeply within each pocket.Importantly, pockets B and F house the primary anchor residues of the peptide. These anchor residues form the main interactions between peptide and HLA, and is suggested to play a key role in stabilising the peptide-HLA complex.


**MHC I pockets**
| Pocket |Residues                            |                                                                                   
|--------|-----------------------------------------------|
| A      | 5, 7, 59, 63, 66, 159, 163, 167, 171                                     |
| B      | 7, 9, 24, 34, 45, 63, 66, 67, 70, 99                |
| C      | 9, 70, 73, 74, 97                        |
| D      | 99, 114, 155, 156, 159, 160                        |
| E      | 97, 114, 147, 152, 156  |
| F      | 77, 80, 81, 84, 95, 123, 143, 146, 147      |


**MHC Class II**: HLA class II molecules consist of two polypeptidic chains, alpha (α) and beta (β), of similar size. The heterodimer HLA molecule forms an extracellular region, a transmembrane region and a cytoplasmic tail. The extracellular region is subdesignated into α1, α2, β1, and β2 domains, that form an immunoglobulin-like domain (α2 and β2) next to the cell membrane, and a membrane– distal peptide-binding cleft (α1 and β1), where the most polymorphic sites of HLA class II molecules of the antigen peptidebinding region (PBR) are located.
Specific binding pockets are formed along the entire length of the peptide-binding cleft of class II molecules, into which anchoring peptides of the antigen can interact with the HLA molecule.


<div style="display: flex; gap: 50px;">

<div>

**HLA-DRB1 and HLA-DQB1 Pockets**

| Pocket | Residues |
|--------|----------|
| P1     | 85, 86, 89, 90 |
| P4     | 13, 26, 28, 70, 74, 78 |
| P6     | 11, 30 |
| P7     | 28, 47, 61, 67, 71 |
| P9     | 9, 37, 57 |

</div>

<div>

**HLA-DQA1 Pockets**

| Pocket | Residues |
|--------|----------|
| P1     | 7, 9, 24, 31, 32, 43, 52 |
| P6     | 11, 62, 65, 66, 69 |
| P9     | 68, 69, 72, 73, 76 |

</div>

</div>



### Functional analysis of HLA variants:
Among the found variants, there are two HLA-A variants that are inside peptide binding pockets and probably affect peptide binding.The F pocket is one of the main pockets in peptide binding so the p.G80R variant probably has more effect.
Also there are two frameshifts in HLA-DRB1 that cause premature stop codon, but they don't change the pockets and have rather high population frequency; which indicates that they probably don't affect HLA function.
| Gene     | Variant (HGVS) | Variant Type         | Location | Protein Effect                | Functional Impact                           | population frequncy|
| -------- | -------------- | -------------------- | -------- | ----------------------------- | -------------------------------------------------- | -------- |
| HLA-DRB1 | c.300_301insAG | Frameshift insertion | Exon 2   | p.R101Sfs*29                  | premature stop codon, not close to peptide binding groove | 0.1407
| HLA-DRB1 | c.294_295del   | Frameshift deletion  | Exon 2   | p.Q99Gfs*28                   | premature stop codon, not close to peptide binding groove | 0.1175                 |
| HLA-A    | c.G238A        | Nonsynonymous SNV    | Exon 2   | p.G80R                        | inside F pocket, probably affects peptide binding| 0.0611
| HLA-A    | c.C290T        | Nonsynonymous SNV    | Exon 2   | p.T97I                        | inside E pocket, probably affects peptide binding(less than p.G80R)|     0.0767  |